# Creating an AI Agent Tutorial

This tutorial will guide you through creating an AI agent using LangGraph and LangChain. We'll build a simple agent that can process user requests through a structured workflow.

## Prerequisites

Make sure you have the following installed:
- Python 3.8+
- Jupyter Notebook
- Required packages (we'll install these in the notebook)

## Setting Up the Environment

In [ ]:
!pip install langchain langgraph langchain-openai python-dotenv

## Importing Required Libraries

In [ ]:
from typing import Dict, List, Any, TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

## Defining the Agent State

First, we need to define the state that our agent will maintain throughout its execution. This state will store messages and track the next steps in the workflow.

In [ ]:
class AgentState(TypedDict):
    """
    Represents the state of the AI agent during execution.

    Attributes:
        messages (List[Dict[str, Any]]): List of messages in the conversation
        next_steps (List[str]): List of next steps to be taken in the workflow
    """
    messages: List[Dict[str, Any]]
    next_steps: List[str]

## Creating the AI Agent Class

Now, let's create the main AI agent class that will handle the workflow and processing of user requests.

In [ ]:
class AIAgent:
    """
    A modular AI agent that processes user requests through a structured workflow.

    This agent uses LangGraph to create a workflow with three main nodes:
    1. Parse Request: Understands the user's input
    2. Think: Plans how to solve the task
    3. Execute: Provides the solution
    """

    def __init__(self, model_name: str = "gpt-3.5-turbo", temperature: float = 0.7):
        """
        Initialize the AI agent.

        Args:
            model_name (str): Name of the language model to use
            temperature (float): Temperature parameter for the language model
        """
        self.llm = ChatOpenAI(
            model=model_name,
            temperature=temperature,
            api_key=os.getenv("OPENAI_API_KEY")
        )
        self.workflow = self._create_workflow()

    def _parse_request(self, state: AgentState) -> AgentState:
        """Parse the user request and determine the task"""
        user_input = state["messages"][-1]["content"]
        prompt = f"Parse this user request into a clear task: {user_input}"
        response = self.llm.invoke(prompt)
        state["messages"].append(
            {"role": "system", "content": f"Parsed task: {response.content}"}
        )
        return state

    def _think(self, state: AgentState) -> AgentState:
        """Think about how to solve the task"""
        task = state["messages"][-1]["content"]
        prompt = f"Think step by step about how to solve this task: {task}"
        response = self.llm.invoke(prompt)
        state["messages"].append(
            {"role": "system", "content": f"Thinking process: {response.content}"}
        )
        return state

    def _execute(self, state: AgentState) -> AgentState:
        """Execute the solution plan"""
        thinking = state["messages"][-1]["content"]
        prompt = f"Based on this thinking, provide a solution: {thinking}"
        response = self.llm.invoke(prompt)
        state["messages"].append({"role": "assistant", "content": response.content})
        return state

    def _decide_next_step(self, state: AgentState) -> str:
        """Decide the next step in the workflow"""
        if len(state["messages"]) <= 1:
            return "parse"
        elif len(state["messages"]) == 2:
            return "think"
        else:
            return "execute"

    def _create_workflow(self) -> StateGraph:
        """Create and configure the agent's workflow graph"""
        workflow = StateGraph(AgentState)

        # Add nodes
        workflow.add_node("parse", self._parse_request)
        workflow.add_node("think", self._think)
        workflow.add_node("execute", self._execute)

        # Add edges
        workflow.add_conditional_edges(START, self._decide_next_step)
        workflow.add_edge("parse", "think")
        workflow.add_edge("think", "execute")
        workflow.add_edge("execute", END)

        return workflow.compile()

    def invoke(self, user_input: str) -> Dict[str, Any]:
        """
        Process a user input through the agent's workflow.

        Args:
            user_input (str): The user's input message

        Returns:
            Dict[str, Any]: The final state of the agent after processing
        """
        initial_state: AgentState = {
            "messages": [{"role": "user", "content": user_input}],
            "next_steps": []
        }
        return self.workflow.invoke(initial_state)

    def get_response(self, result: Dict[str, Any]) -> str:
        """
        Extract the final response from the agent's result.

        Args:
            result (Dict[str, Any]): The result from invoking the agent

        Returns:
            str: The final response message
        """
        return result["messages"][-1]["content"]

## Using the Agent

Now that we have our agent implementation, let's create an instance and try it out:

In [ ]:
# Create a basic agent with default settings
agent = AIAgent()

# Process a user request
result = agent.invoke("Find the best restaurants in New York")

# Get the final response
response = agent.get_response(result)
print(response)

## Understanding the Workflow

Let's look at the complete conversation history to understand how the agent processed the request:

In [ ]:
# Print the complete conversation
for message in result["messages"]:
    print(f"{message['role'].upper()}: {message['content']}")
    print("---")

## Customizing the Agent

You can customize the agent in several ways:

1. Use different models:
```python
agent = AIAgent(model_name="gpt-4")
```

2. Adjust the temperature for different response styles:
```python
agent = AIAgent(temperature=0.2)  # More focused and deterministic
agent = AIAgent(temperature=1.0)  # More creative and varied
```

3. Create multiple agents for different purposes:
```python
creative_agent = AIAgent(temperature=0.9)
precise_agent = AIAgent(temperature=0.1)
```

## Example: Comparing Different Agent Configurations

Let's see how different configurations affect the agent's responses:

In [ ]:
# Create agents with different configurations
creative_agent = AIAgent(temperature=0.9)
precise_agent = AIAgent(temperature=0.1)

# Test with the same prompt
prompt = "Write a short story about a robot learning to paint"

print("Creative Agent Response:")
result = creative_agent.invoke(prompt)
print(creative_agent.get_response(result))
print("\n---\n")

print("Precise Agent Response:")
result = precise_agent.invoke(prompt)
print(precise_agent.get_response(result))

## Next Steps

You can extend this agent by:
1. Adding more specialized nodes for different types of tasks
2. Integrating with external APIs for real-time data
3. Adding memory to maintain context across conversations
4. Implementing error handling and retry mechanisms

## Conclusion

You've now learned how to create and use a modular AI agent! The agent can process user requests through a structured workflow, and you can customize its behavior by adjusting parameters like the model and temperature.